# Script de Inyeccion de datos
En este archivo, inyectamos los datos homologados y normalizados

### IMPORTACIONES DE BIBLIOTECAS NECESARIAS

In [7]:
import polars as pl
import sqlalchemy as sa
from IPython.display import display
from pydantic import BaseModel

### Constantes del Notebook
Aqui se definen las constantes que se necesiten para subir los archivos

In [ ]:
FILE_PATH = './layouts/oferta_academicav2.3.xlsx'  # archivo que se desea leer
SHEETS_NAMES = ['carrera_habilidad']  # solo se procesa esta tabla
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = "password"
DB_CONECTION = "athena"
DIR_SAVE = "load_data.sql"

### Modelos de pydantic

en este apartado estan los modelos para subir a la base de datos los datos correspondientes sin romper la tabla

In [12]:
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Escuela(BaseModel):
    id_escuela: int
    nombre_escuela: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str
    id_area: int

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class criterios(BaseModel):
    id_criterio: int
    descripcion: str
    id_habilidad: int

### Funciones de procesamiento

aqui estan todas las funciones para procesar el excel, dependiendo de la hoja lo procesa de manera distinta

In [9]:
class SQLGenerator:
    def carrera(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO carrera (id_carrera, nombre_carrera, id_area) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_carrera']}, '{row['nombre_carrera']}', {row['id_area']})")
        sql += ",\n".join(values) + ";"
        return sql

    def area(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO area (id_area, nombre_area) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_area']}, '{row['nombre_area']}')")
        sql += ",\n".join(values) + ";"
        return sql

    def escuela(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO escuela (id_escuela, nombre_escuela) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_escuela']}, '{row['nombre_escuela']}')")
        sql += ",\n".join(values) + ";"
        return sql

    def habilidad(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO habilidad (id_habilidad, descripcion) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_habilidad']}, '{row['descripcion']}')")
        sql += ",\n".join(values) + ";"
        return sql

    def carrera_habilidad(self, df: pl.DataFrame) -> str:
        sql = """BEGIN;

DROP TABLE IF EXISTS public.carrera_habilidad;

CREATE TABLE public.carrera_habilidad (
    id_carrera INTEGER NOT NULL,
    id_habilidad INTEGER NOT NULL,
    CONSTRAINT pk_carrera_habilidad PRIMARY KEY (id_carrera, id_habilidad),
    CONSTRAINT fk_carrera_habilidad_carrera
        FOREIGN KEY (id_carrera) REFERENCES public.carrera (id_carrera),
    CONSTRAINT fk_carrera_habilidad_habilidad
        FOREIGN KEY (id_habilidad) REFERENCES public.habilidad (id_habilidad)
);

INSERT INTO public.carrera_habilidad (id_carrera, id_habilidad) VALUES """
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_carrera']}, {row['id_habilidad']})")
        sql += ",\n".join(values) + ";\n\nCOMMIT;"
        return sql

    def criterio(self, df: pl.DataFrame) -> str:
        sql = "INSERT INTO criterio (id_criterio, descripcion, id_habilidad) VALUES "
        values = []
        for row in df.iter_rows(named=True):
            values.append(f"({row['id_criterio']}, '{row['nombre_criterio']}', {row['id_habilidad']})")
        sql += ",\n".join(values) + ";"
        return sql

def db_conection():
    try:
        engine = sa.create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_CONECTION}')
        print("Database connection successful.")
        return engine
    except Exception as e:
        print(f"Error connecting to the database: {e}")
        return None

def read_excel_file(file_path: str, sheet_name: str) -> pl.DataFrame:
    df_sheet = pl.read_excel(file_path, sheet_name=sheet_name)
    return df_sheet

def view_excel_sheets(df: pl.DataFrame, sheet_name: str):
    print(f"Sheet: {sheet_name}")
    display(df[:5])

def ask_submit_data(df: pl.DataFrame, sheet_name: str):
    SQLGen = SQLGenerator()
    sql_archive = getattr(SQLGen, sheet_name)(df)
    print(f"SQL for {sheet_name}:\n{sql_archive}")
    return sql_archive

def save_text_to_file(text: str, filename: str):
    with open(filename, 'w') as f:
        f.write(text)

In [11]:
def execute_sql_in_database(sql_text: str) -> None:
    engine = db_conection()
    if engine is None:
        raise RuntimeError("No fue posible conectar con la base de datos.")

    statements = [statement.strip() for statement in sql_text.split(";") if statement.strip()]
    with engine.begin() as connection:
        for statement in statements:
            if statement.upper() not in {"BEGIN", "COMMIT"}:
                connection.exec_driver_sql(statement)

    with engine.connect() as connection:
        count = connection.execute(
            sa.text("SELECT COUNT(*) FROM public.carrera_habilidad")
        ).scalar_one()
    print(f"Registros en carrera_habilidad: {count}")

### Ejecucion

aqui se ejecuta el script, haciendo que se procese y se suba la informacion

In [12]:
if __name__ == "__main__":
    sql_data = []
    for sheet_name in SHEETS_NAMES:
        df = read_excel_file(FILE_PATH, sheet_name)
        view_excel_sheets(df, sheet_name)
        sql_text = ask_submit_data(df, sheet_name)
        sql_data.append(sql_text)

    complete_sql = "\n\n".join(sql_data)
    save_text_to_file(complete_sql, DIR_SAVE)
    execute_sql_in_database(complete_sql)

Sheet: carrera_habilidad


C:\Users\emanu\AppData\Local\Temp\ipykernel_22532\1617049969.py:74: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df_sheet = pl.read_excel(file_path, sheet_name=sheet_name)


id_carrera,id_habilidad
i64,i64
1,1
1,2
1,3
1,4
1,5


SQL for carrera_habilidad:
BEGIN;

DROP TABLE IF EXISTS public.carrera_habilidad;

CREATE TABLE public.carrera_habilidad (
    id_carrera INTEGER NOT NULL,
    id_habilidad INTEGER NOT NULL,
    CONSTRAINT pk_carrera_habilidad PRIMARY KEY (id_carrera, id_habilidad),
    CONSTRAINT fk_carrera_habilidad_carrera
        FOREIGN KEY (id_carrera) REFERENCES public.carrera (id_carrera),
    CONSTRAINT fk_carrera_habilidad_habilidad
        FOREIGN KEY (id_habilidad) REFERENCES public.habilidad (id_habilidad)
);

INSERT INTO public.carrera_habilidad (id_carrera, id_habilidad) VALUES (1, 1),
(1, 2),
(1, 3),
(1, 4),
(1, 5),
(1, 6),
(1, 7),
(40, 7),
(40, 8),
(40, 9),
(40, 10),
(40, 11),
(40, 12),
(41, 6),
(41, 13),
(41, 14),
(41, 15),
(41, 16),
(41, 17),
(7, 18),
(7, 19),
(7, 20),
(7, 21),
(7, 22),
(7, 23),
(10, 12),
(10, 22),
(10, 24),
(10, 25),
(10, 26),
(10, 27),
(11, 28),
(11, 29),
(11, 30),
(11, 31),
(11, 32),
(12, 7),
(12, 21),
(12, 33),
(12, 34),
(12, 35),
(8, 23),
(8, 35),
(8, 36),
(8, 3